# Medical RAG Evaluation

End-to-end runner for all three phases:
1. **Phase 1a** — No-retrieval baseline (LLM answers blind)
2. **Phase 1b** — Vanilla dense RAG on PubMedQA
3. **Phase 2** — Hybrid BM25 + dense retrieval
4. **Phase 3** — Failure-mode analysis of error cases

In [1]:
import sys
sys.path.insert(0, '..')

from src.pipeline import run_full_pipeline

In [2]:
summary, baseline_results, dense_results, hybrid_results = run_full_pipeline(sample_size=200)

Loading PubMedQA...


Loaded 200 questions.

Building corpus...
Corpus size: 676 passages.

Building dense index...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/11 [00:00<?, ?it/s]


Building BM25 index...

Building hybrid retriever...

--- Phase 1a: No-retrieval baseline ---


...


--- Phase 1b: Dense RAG ---


RAG (dense): 100%|██████████████████████████| 200/200 [1:25:39<00:00, 25.70s/it]



--- Phase 2: Hybrid RAG ---


RAG (hybrid): 100%|█████████████████████████| 200/200 [2:54:32<00:00, 52.36s/it]


=== Results ===
  No retrieval          Accuracy: 51.50%  Hit-rate: 0.0
  Dense RAG             Accuracy: 68.50%  Hit-rate: 0.995
  Hybrid RAG            Accuracy: 67.50%  Hit-rate: 0.98


In [3]:
import json
print(json.dumps(summary, indent=2))

{
  "sample_size": 200,
  "corpus_size": 676,
  "baseline": {
    "label": "No retrieval",
    "n_questions": 200,
    "accuracy": 0.515,
    "hit_rate": 0.0
  },
  "dense_rag": {
    "label": "Dense RAG",
    "n_questions": 200,
    "accuracy": 0.685,
    "hit_rate": 0.995
  },
  "hybrid_rag": {
    "label": "Hybrid RAG",
    "n_questions": 200,
    "accuracy": 0.675,
    "hit_rate": 0.98
  }
}


## Phase 3: Failure-Mode Analysis

Identify questions the hybrid system got wrong and categorise the failure mode.

In [4]:
# Collect errors from hybrid RAG (the best system)
errors = [r for r in hybrid_results if r['predicted'] != r['gold_answer']]
print(f"Total errors: {len(errors)}")

# Show first 15 for manual analysis
for i, err in enumerate(errors[:15]):
    hit = err['retrieval_hit']
    print(f"\n--- Error {i+1} ---")
    print(f"PubID: {err['pubid']}")
    print(f"Question: {err['question'][:120]}...")
    print(f"Gold: {err['gold_answer']}  |  Predicted: {err['predicted']}")
    print(f"Retrieval hit: {hit}")
    if hit:
        print("  -> Category: Retrieval hit, wrong generation")
    else:
        print("  -> Category: Retrieval miss")

Total errors: 65

--- Error 1 ---
PubID: 15483019
Question: Is eligibility for a chemotherapy protocol a good prognostic factor for invasive bladder cancer after radical cystectomy...
Gold: yes  |  Predicted: no
Retrieval hit: True
  -> Category: Retrieval hit, wrong generation

--- Error 2 ---
PubID: 9582182
Question: Does the SCL 90-R obsessive-compulsive dimension identify cognitive impairments?...
Gold: yes  |  Predicted: no
Retrieval hit: True
  -> Category: Retrieval hit, wrong generation

--- Error 3 ---
PubID: 25636371
Question: Is it possible to stop treatment with nucleos(t)ide analogs in patients with e-antigen negative chronic hepatitis B?...
Gold: maybe  |  Predicted: yes
Retrieval hit: True
  -> Category: Retrieval hit, wrong generation

--- Error 4 ---
PubID: 25987398
Question: The influence of atmospheric pressure on aortic aneurysm rupture--is the diameter of the aneurysm important?...
Gold: maybe  |  Predicted: yes
Retrieval hit: True
  -> Category: Retrieval hit, wro